# T5 Fine-Tuning for Sentence Simplification

**Purpose:** Overview of the training procedure (thesis Section *Training*).

**Runnable code:** `training/train.py` — this notebook is for the committee; run training from the terminal.

| Component | Choice |
|-----------|--------|
| Base model | `t5-base` |
| Task prefix | `summarize: ` + source sentence |
| Max length | 128 tokens |
| Optimizer | AdamW, weight decay 0.01 |
| Selection | Lowest **eval_loss**, early stopping patience 4 |

## 1. Three training runs (thesis Table)

- **Run 1:** WikiLarge (CEFR Δ > 0.8) + ASSET + GPT — lr 5e-5, warmup 500, ~17k pairs
- **Run 2:** ASSET + GPT only — lr 3e-5, warmup 50, ~3.4k pairs
- **Run 3:** WikiLarge (CEFR Δ > 1.3) + ASSET + GPT — lr 3e-5, warmup 100, ~7.3k pairs

Data preparation is separate; training only reads a filtered CSV (`original`, `simplified`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from config import RUNS, TASK_PREFIX, BASE_MODEL

for key, cfg in RUNS.items():
    print(f"{key}: lr={cfg.learning_rate}, warmup={cfg.warmup_steps} — {cfg.description}")
print(f"\nBase: {BASE_MODEL}, prefix: {TASK_PREFIX!r}")

## 2. CEFR metrics during training

- `cefr_diff` — mean readability reduction (continuous CEFR)
- `cefr_close1` — share with |diff − 1| < 0.5
- `sim_orig` — SequenceMatcher vs source

Eval decode: greedy argmax (not beam search).

## 3. Train & evaluate

```bash
python training/train.py --run run3 --data-csv YOUR.csv --output-dir ./t5-base-simplification-res4
python training/evaluate_checkpoint.py --model-dir ./t5-base-simplification-res4
```

Legacy Colab log: `Untitled30.ipynb` (see `training/TRAINING_LEGACY.md`).